# Run 1c — YOLO11m Baseline (6 clases)

**Epic:** TTV-118

Mismo dataset v7, misma config baseline, pero filtrando a 6 clases core.

**Excluidas:** `bicycle_text`, `clothes_text`, `helmet_text`, `objects`

**Hipótesis:** Sin clases ruidosas/minoritarias, mAP global supera 0.80

---

In [ ]:
import torch

assert torch.cuda.is_available(), "ERROR: No GPU. Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q roboflow ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/cycling-photo-ai/experiments'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "xOdnFACkI2vaUzBKVRic"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("titan-ca4ce").project("titan-detection-jedpa")
version = project.version(7)

dataset = version.download("yolov11", location="/content/dataset_v1")
print("Dataset descargado")

## Filtrar a 6 clases

Original (10 clases):
```
0: bicycle
1: bicycle_text      ← EXCLUIR
2: clothes_text       ← EXCLUIR
3: competidor_number
4: cyclist
5: cyclist_clothes
6: cyclist_with_bike
7: helmet
8: helmet_text        ← EXCLUIR
9: objects            ← EXCLUIR
```

Nuevo (6 clases):
```
0: bicycle
1: competidor_number
2: cyclist
3: cyclist_clothes
4: cyclist_with_bike
5: helmet
```

In [ ]:
from pathlib import Path
import shutil

# Clases a mantener: ID original → nuevo ID
KEEP_CLASSES = {
    0: 0,   # bicycle → 0
    3: 1,   # competidor_number → 1
    4: 2,   # cyclist → 2
    5: 3,   # cyclist_clothes → 3
    6: 4,   # cyclist_with_bike → 4
    7: 5,   # helmet → 5
}
EXCLUDE_IDS = {1, 2, 8, 9}  # bicycle_text, clothes_text, helmet_text, objects

src_dir = Path("/content/dataset_v1")
filtered_dir = Path("/content/dataset_v1_6classes")

# Copiar estructura
if filtered_dir.exists():
    shutil.rmtree(filtered_dir)

total_removed = 0
total_kept = 0

for split in ["train", "valid", "test"]:
    # Copiar imágenes
    src_imgs = src_dir / split / "images"
    dst_imgs = filtered_dir / split / "images"
    dst_labels = filtered_dir / split / "labels"
    dst_imgs.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    for img_file in src_imgs.glob("*.*"):
        shutil.copy2(img_file, dst_imgs / img_file.name)

    # Filtrar labels
    src_labels = src_dir / split / "labels"
    for label_file in src_labels.glob("*.txt"):
        new_lines = []
        for line in label_file.read_text().strip().split("\n"):
            if not line.strip():
                continue
            parts = line.strip().split()
            cls_id = int(parts[0])
            if cls_id in KEEP_CLASSES:
                parts[0] = str(KEEP_CLASSES[cls_id])
                new_lines.append(" ".join(parts))
                total_kept += 1
            else:
                total_removed += 1
        (dst_labels / label_file.name).write_text("\n".join(new_lines))

print(f"Annotations kept: {total_kept}")
print(f"Annotations removed: {total_removed}")
print(f"Removal rate: {total_removed / (total_kept + total_removed) * 100:.1f}%")

In [ ]:
# Crear data.yaml para 6 clases
data_yaml_content = """train: ../train/images
val: ../valid/images
test: ../test/images

nc: 6
names: ['bicycle', 'competidor_number', 'cyclist', 'cyclist_clothes', 'cyclist_with_bike', 'helmet']
"""

(filtered_dir / "data.yaml").write_text(data_yaml_content)
print("data.yaml creado:")
print(data_yaml_content)

# Verificar
for split in ["train", "valid", "test"]:
    imgs = list((filtered_dir / split / "images").glob("*.*"))
    lbls = list((filtered_dir / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")

In [ ]:
# Verificar distribución de clases
class_names_6 = ['bicycle', 'competidor_number', 'cyclist', 'cyclist_clothes', 'cyclist_with_bike', 'helmet']

for split in ["train", "valid", "test"]:
    counts = {i: 0 for i in range(6)}
    for lf in (filtered_dir / split / "labels").glob("*.txt"):
        for line in lf.read_text().strip().split("\n"):
            if line.strip():
                counts[int(line.split()[0])] += 1
    print(f"\n{split}:")
    for i, name in enumerate(class_names_6):
        print(f"  {name:25s} {counts[i]}")

In [ ]:
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print(f"Seed: {SEED}")

## Entrenar YOLO11m — Run 1c (6 clases)

Misma config que Run 1 baseline. Única diferencia: 6 clases en vez de 10.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "run1c_yolo11m_6classes"

model = YOLO("yolo11m.pt")

results = model.train(
    data=str(filtered_dir / "data.yaml"),
    epochs=200,
    patience=30,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=7.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.0,
    cutmix=0.0,
    cls_pw=1.0,
    save_json=True,
    deterministic=True,
    seed=SEED,
    project="/content/experiments",
    name=RUN_NAME,
)

In [ ]:
import pandas as pd

run_dir = Path(f"/content/experiments/{RUN_NAME}")

print("=== Métricas finales ===")
for key, val in results.results_dict.items():
    print(f"  {key}: {val:.4f}")

results_csv = run_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"\nEpochs entrenados: {len(df)}")
    print(f"Mejor mAP@0.5: {df['metrics/mAP50(B)'].max():.4f} (epoch {df['metrics/mAP50(B)'].idxmax()})")
    print(f"Mejor mAP@0.5:0.95: {df['metrics/mAP50-95(B)'].max():.4f} (epoch {df['metrics/mAP50-95(B)'].idxmax()})")

In [ ]:
import matplotlib.pyplot as plt

if results_csv.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(df['train/box_loss'], label='train box')
    axes[0].plot(df['train/cls_loss'], label='train cls')
    axes[0].plot(df['val/box_loss'], label='val box', linestyle='--')
    axes[0].plot(df['val/cls_loss'], label='val cls', linestyle='--')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].set_xlabel('Epoch')

    axes[1].plot(df['metrics/mAP50(B)'], label='mAP@0.5')
    axes[1].plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axes[1].axhline(y=0.80, color='r', linestyle=':', label='Target 0.80')
    axes[1].set_title('mAP')
    axes[1].legend()
    axes[1].set_xlabel('Epoch')

    axes[2].plot(df['metrics/precision(B)'], label='Precision')
    axes[2].plot(df['metrics/recall(B)'], label='Recall')
    axes[2].set_title('Precision / Recall')
    axes[2].legend()
    axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(run_dir / 'training_curves.png', dpi=150)
    plt.show()

In [ ]:
best_model = YOLO(str(run_dir / "weights" / "best.pt"))
val_results = best_model.val(data=str(filtered_dir / "data.yaml"), imgsz=640, save_json=True)

class_names_6 = ['bicycle', 'competidor_number', 'cyclist', 'cyclist_clothes', 'cyclist_with_bike', 'helmet']

print("\n=== Per-class AP@0.5 ===")
for i, name in enumerate(class_names_6):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    ap = val_results.box.ap[i] if i < len(val_results.box.ap) else 0
    print(f"  {name:25s} AP@0.5={ap50:.4f}  AP@0.5:0.95={ap:.4f}")

In [ ]:
from IPython.display import Image, display

for fname, title in [("confusion_matrix_normalized.png", "Confusion Matrix"),
                      ("PR_curve.png", "PR Curves"),
                      ("val_batch0_pred.jpg", "Sample predictions")]:
    fpath = run_dir / fname
    if fpath.exists():
        print(f"\n{title}:")
        display(Image(filename=str(fpath), width=800))

In [ ]:
import shutil

drive_run_dir = Path(DRIVE_OUTPUT) / RUN_NAME
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(run_dir, drive_run_dir)

weights_size = (drive_run_dir / "weights" / "best.pt").stat().st_size / 1e6
print(f"Guardado en: {drive_run_dir}")
print(f"best.pt: {weights_size:.1f} MB")

In [ ]:
print("="*60)
print("RESUMEN PARA EXPERIMENT_LOG.md")
print("="*60)
print(f"\n### Run 1c — YOLO11m Baseline (6 clases)")
print(f"- **Fecha:** {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print(f"- **Config:** baseline (mismo que Run 1)")
print(f"- **Dataset:** v1, filtrado a 6 clases (sin *_text, sin objects)")
print(f"- **GPU:** {torch.cuda.get_device_name(0)}")
print(f"- **Clases:** bicycle, competidor_number, cyclist, cyclist_clothes, cyclist_with_bike, helmet")
print(f"- **Annotations removed:** {total_removed} ({total_removed / (total_kept + total_removed) * 100:.1f}%)")
print(f"- **Epochs entrenados:** {len(df)}")
print(f"- **Mejor epoch:** {df['metrics/mAP50(B)'].idxmax()}")
print(f"")
print(f"| Métrica | Run 1c (6cls) | Run 1 (10cls) | Δ |")
print(f"|---|---|---|---|")
print(f"| mAP@0.5 | {df['metrics/mAP50(B)'].max():.4f} | 0.7223 | {df['metrics/mAP50(B)'].max() - 0.7223:+.4f} |")
print(f"| mAP@0.5:0.95 | {df['metrics/mAP50-95(B)'].max():.4f} | 0.5113 | {df['metrics/mAP50-95(B)'].max() - 0.5113:+.4f} |")
print(f"| Precision | {df['metrics/precision(B)'].max():.4f} | 0.8020 | {df['metrics/precision(B)'].max() - 0.8020:+.4f} |")
print(f"| Recall | {df['metrics/recall(B)'].max():.4f} | 0.7362 | {df['metrics/recall(B)'].max() - 0.7362:+.4f} |")
print(f"")
print(f"**Per-class AP@0.5:**")
print(f"")
print(f"| Clase | AP@0.5 |")
print(f"|---|---|")
for i, name in enumerate(class_names_6):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    print(f"| {name} | {ap50:.4f} |")
print(f"\nPesos guardados en: {drive_run_dir}/weights/best.pt")